# 🔎 InternLens
### *See Beyond the Job Description.*

Chatbot berbasis LLM (Groq API + Python) untuk membantu mahasiswa dan pencari magang **menganalisis job/internship description sebelum melamar**.

**Tiga fungsi utama:**
- 🚩 **Red Flag Scan**: mengidentifikasi hal yang jelas, tidak jelas, dan perlu diklarifikasi
- 🎯 **Skill Match**: mencocokkan profil user dengan job description
- ❓ **Questions to Ask Recruiter**: pertanyaan yang relevan untuk recruiter

**Cara menjalankan:** jalankan cell dari atas ke bawah. Chatbot interaktif berada di cell paling akhir.

**Catatan:** InternLens hanya menganalisis informasi yang diberikan user dan tidak menyimpulkan bahwa suatu lowongan pasti aman atau berbahaya.

## Step 1: Setup dan Instalasi Library

**Tujuan:** Memasang dan mengimpor library yang dibutuhkan.

**Cara kerja:** `pip install` memasang SDK Groq. Library lain (`json`, `datetime`, `getpass`) sudah bawaan Python.

**Hubungannya dengan tugas:** Berbasis LLM dan menggunakan Groq API (bukan model lokal).

In [ ]:
# 1. Install library Groq (jalankan sekali di Colab)
!pip install -q groq

# Import library yang dibutuhkan
import json                      # menyimpan dan membaca riwayat percakapan
from datetime import datetime    # timestamp untuk nama file
from getpass import getpass      # input API key tanpa menampilkannya di layar
from groq import Groq            # SDK resmi Groq

## Step 2: API Key dengan Aman

**Tujuan:** Mengambil API key tanpa menuliskannya di kode.

**Cara kerja:** Kode mencoba membaca secret `GROQ_API_KEY` dari Colab. Jika gagal (misalnya di Jupyter lokal), kode meminta input lewat `getpass`. Yang dicetak hanya status, bukan key-nya.

**Hubungannya dengan tugas:** API key tidak di-hardcode dan tidak ditampilkan di output.

> Di Colab: simpan key pada ikon 🔑 **Secrets** (sidebar kiri) dengan nama `GROQ_API_KEY`, lalu aktifkan *Notebook access*.

In [ ]:
# 2. Load API key
# Prioritas 1: Colab Secrets. Prioritas 2: input manual lewat getpass.
try:
    from google.colab import userdata
    api_key = userdata.get("GROQ_API_KEY")
except Exception:
    api_key = getpass("Masukkan GROQ_API_KEY: ")

# Pastikan API key tidak kosong (tanpa mencetak isinya)
if not api_key:
    raise ValueError("API key belum diisi.")
print("✅ API key berhasil dimuat.")

## Step 3: Groq Client

**Tujuan:** Membuat objek `client` dan menyimpan konfigurasi model.

**Cara kerja:** `Groq(api_key=...)` menyiapkan koneksi ke API. `MODEL_NAME` dan `TEMPERATURE` dipakai di setiap request.

**Hubungannya dengan tugas:** Groq API, bukan model lokal. Temperature 0.3 dipilih karena analisis membutuhkan jawaban yang konsisten.

> Model `llama-3.3-70b-versatile` sudah dijadwalkan berhenti pada 16 Agustus 2026, sehingga notebook ini memakai `openai/gpt-oss-120b`. Jika muncul error model tidak ditemukan, cek daftar model terbaru di `console.groq.com/docs/models` lalu ubah `MODEL_NAME`.

In [ ]:
# 3. Membuat client untuk berkomunikasi dengan Groq API
client = Groq(api_key=api_key)

# Konfigurasi model (mudah diganti bila model berubah)
MODEL_NAME = "openai/gpt-oss-120b"
TEMPERATURE = 0.3   # rendah = jawaban lebih konsisten, cocok untuk analisis

## Step 4: System Prompt InternLens

**Tujuan:** Mendefinisikan peran, aturan, dan format jawaban chatbot.

**Cara kerja:** Teks ini dikirim sebagai pesan `system` pada setiap request, sehingga model selalu mengikuti aturan yang sama (netral, tidak menuduh, tidak memberi skor).

**Hubungannya dengan tugas:** Memenuhi requirement system prompt, serta fitur Red Flag Scan, Skill Match, dan Questions to Ask Recruiter.

In [ ]:
# 4. System prompt InternLens
SYSTEM_PROMPT = """
Kamu adalah InternLens, seorang AI Internship Analysis Assistant.
Slogan: "See Beyond the Job Description."

TUGAS UTAMA
1. Menganalisis job/internship description yang diberikan user.
2. Mengidentifikasi informasi yang sudah jelas dan yang belum tersedia.
3. Mengidentifikasi potential concerns berdasarkan teks yang diberikan.
4. Menjelaskan ALASAN dari setiap concern (bukan sekadar label).
5. Mencocokkan job description dengan skill user jika profil diberikan.
6. Membuat 3-7 pertanyaan yang sebaiknya ditanyakan kepada recruiter.

ATURAN WAJIB
- Analisis HANYA berdasarkan teks yang diberikan user. Jangan mengarang informasi
  yang tidak ada pada job description.
- Jika informasi tidak tersedia, katakan "tidak disebutkan" atau minta user
  memberikan job description yang lebih lengkap.
- Jangan menuduh perusahaan melakukan penipuan atau menyebut lowongan "scam" atau "buruk".
- Jangan menyatakan lowongan pasti aman atau pasti berbahaya.
- Jangan memberikan skor (misalnya 85/100), ranking perusahaan, atau keputusan final.
- Gunakan bahasa objektif: "perlu diklarifikasi", "belum dijelaskan", "berpotensi".
- Untuk Skill Match, jangan mengatakan user "pasti cocok" atau "tidak cocok".
  Gunakan kategori: Strong Match, Partial Match, Skill Gap, Need Clarification.
- Gunakan profil/skill user dari pesan sebelumnya dalam percakapan jika ada.
- Sertakan disclaimer singkat bahwa analisis hanya berdasarkan informasi yang diberikan user.

BAHASA
Bahasa Indonesia yang sederhana, profesional, dan mudah dipahami mahasiswa.
Istilah teknis umum (job scope, allowance, deliverable) boleh tetap dalam bahasa Inggris.

FORMAT ANALISIS (gunakan saat user memberikan job description)
## 🔎 Internship Analysis
### 📌 Role Overview
Job title, main responsibilities, work arrangement, working hours, compensation.
### 🟢 Clear Information
### 🟡 Missing / Unclear Information
### 🟠 Potential Concerns
Untuk setiap concern tulis: **Concern:**, **Why:**, **What to clarify:**
### 🎯 Skill Match
Jika profil user tersedia: Strong Match, Partial Match, Skill Gap, Need Clarification.
Jika belum ada, tulis: "Berikan skill atau background kamu jika ingin melakukan skill matching."
### ❓ Questions for Recruiter
3-7 pertanyaan berdasarkan informasi yang benar-benar kurang atau ambigu.
### 📋 Overall Takeaway
Ringkasan objektif tanpa skor dan tanpa keputusan final.

LINGKUP
Tetap dalam konteks internship, rekrutmen, persiapan karier, dan analisis job description.
Jika user bertanya di luar konteks, jawab singkat lalu arahkan kembali ke fungsi utama InternLens.
Jika user hanya memberikan profil/skill, konfirmasi bahwa profil diterima dan
minta user mengirim job description.
"""

## Step 5: Conversation History

**Tujuan:** Menyiapkan list `messages` sebagai memori percakapan.

**Cara kerja:** Setiap pesan user dan jawaban asisten akan ditambahkan ke list ini, lalu seluruh list dikirim ke API pada setiap request. `reset_riwayat()` dipakai oleh command `clear`.

**Hubungannya dengan tugas:** Conversation history dan chatbot mengingat konteks dalam satu sesi.

In [ ]:
# 5. Riwayat percakapan dimulai dengan system prompt
messages = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def reset_riwayat(messages):
    """Mengosongkan riwayat, lalu mengisi ulang hanya dengan system prompt."""
    messages.clear()
    messages.append({"role": "system", "content": SYSTEM_PROMPT})

## Step 6: Fungsi Request ke Groq + Error Handling

**Tujuan:** Mengirim request ke Groq dan mengembalikan teks jawaban.

**Cara kerja:** `client.chat.completions.create()` mengirim seluruh `messages`. Jika terjadi error (koneksi, key salah, model tidak tersedia), blok `except` menampilkan pesan ramah dan mengembalikan `None`, sehingga program tidak crash.

**Hubungannya dengan tugas:** Error handling. Fungsi ini juga menjadi dasar sebelum streaming ditambahkan.

In [ ]:
# 6. Fungsi request ke Groq API (tanpa streaming)
def kirim_pesan(messages):
    """Mengirim riwayat ke Groq. Mengembalikan jawaban, atau None jika gagal."""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=TEMPERATURE,
        )
        return response.choices[0].message.content
    except Exception as e:
        print("⚠️ Maaf, terjadi masalah saat menghubungi Groq API. Silakan coba lagi.")
        print(f"   Detail teknis: {e}")
        return None

In [ ]:
# Uji cepat koneksi (opsional)
print(kirim_pesan([{"role": "user", "content": "Balas dengan satu kata: siap"}]))

## Step 7: Streaming Response

**Tujuan:** Menampilkan jawaban sedikit demi sedikit di console.

**Cara kerja:** Dengan `stream=True`, API mengirim potongan teks (*chunk*). Setiap potongan dicetak langsung dan ditambahkan ke `jawaban_lengkap`, yang dikembalikan di akhir agar bisa masuk ke riwayat.

**Hubungannya dengan tugas:** Bonus streaming response.

> Model ini melakukan *reasoning* terlebih dahulu, sehingga teks bisa muncul dengan sedikit jeda di awal.

In [ ]:
# 7. Fungsi request dengan streaming
def kirim_pesan_streaming(messages):
    """Mengirim riwayat ke Groq dengan streaming. Mengembalikan jawaban lengkap, atau None jika gagal."""
    try:
        stream = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=TEMPERATURE,
            stream=True,              # jawaban datang bertahap
        )

        jawaban_lengkap = ""
        for chunk in stream:
            if not chunk.choices:
                continue
            potongan = chunk.choices[0].delta.content
            if potongan:                                   # bisa None, jadi dicek dulu
                print(potongan, end="", flush=True)        # tampilkan langsung
                jawaban_lengkap += potongan                # kumpulkan untuk riwayat
        print()

        if not jawaban_lengkap:
            raise ValueError("Respons kosong.")
        return jawaban_lengkap

    except Exception as e:
        print("\n⚠️ Maaf, terjadi masalah saat menghubungi Groq API. Silakan coba lagi.")
        print(f"   Detail teknis: {e}")
        return None

## Step 8: Save dan Load Riwayat

**Tujuan:** Menyimpan dan membaca kembali percakapan dalam format JSON.

**Cara kerja:** `simpan_riwayat` menulis list `messages` ke file bernama `riwayat_internlens_YYYYMMDD_HHMMSS.json`. `muat_riwayat` membaca file tersebut. API key tidak ikut tersimpan karena tidak pernah masuk ke `messages`.

**Hubungannya dengan tugas:** Bonus save history.

In [ ]:
# 8. Save dan load riwayat percakapan
def simpan_riwayat(messages):
    """Menyimpan riwayat ke file JSON dengan timestamp pada nama file."""
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        nama_file = f"riwayat_internlens_{timestamp}.json"
        with open(nama_file, "w", encoding="utf-8") as f:
            json.dump(messages, f, ensure_ascii=False, indent=2)
        print(f"💾 Riwayat disimpan ke: {nama_file}")
        return nama_file
    except Exception as e:
        print(f"⚠️ Gagal menyimpan riwayat: {e}")
        return None

def muat_riwayat(nama_file):
    """Membaca riwayat dari file JSON. Mengembalikan list messages, atau None jika gagal."""
    try:
        with open(nama_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            raise ValueError("Format file tidak sesuai.")
        return data
    except Exception as e:
        print(f"⚠️ Gagal memuat riwayat: {e}")
        return None

## Step 9: Command Handling

**Tujuan:** Menangani perintah khusus di luar percakapan biasa.

**Cara kerja:** Input dicocokkan dengan daftar command. Jika cocok, aksinya dijalankan dan fungsi mengembalikan status. Jika tidak cocok, input dianggap pesan biasa untuk chatbot. Command `contoh` memudahkan demo saat presentasi.

**Hubungannya dengan tugas:** Minimal 2 command (`exit`, `clear`), ditambah `save` dan `help` sebagai bonus. Command `load` dan `contoh` menjadi fitur tambahan yang relevan.

In [ ]:
# 9. Command handling

# Contoh job description untuk demo (dipakai command "contoh" dan Step 11)
CONTOH_JOB_DESC = """Tolong analisis job description berikut:

Data Analyst Intern

Responsibilities:
* Analyze company data
* Create reports
* Manage social media
* Contact clients
* Assist marketing activities
* Create presentations
* Support administrative tasks

Requirements:
* Python
* SQL
* Power BI
* Photoshop
* Video editing
* Digital marketing
* Excellent communication
* Available 40 hours/week

Benefits:
* Valuable experience
* Career opportunity
* Certificate"""

BANNER = """==================================================
              INTERNLENS
       See Beyond the Job Description
=================================================="""

def tampilkan_bantuan():
    print("""
Commands:
  exit    - Keluar dari chatbot
  clear   - Memulai percakapan baru
  save    - Menyimpan riwayat percakapan
  load    - Memuat riwayat (contoh: load riwayat_internlens_xxx.json)
  contoh  - Menganalisis contoh job description Data Analyst Intern
  help    - Menampilkan bantuan

Cara menggunakan:
  1. (Opsional) Ceritakan background dan skill kamu.
  2. Tempel job/internship description yang ingin dianalisis.
  3. InternLens akan memberikan analisis, skill match, dan pertanyaan untuk recruiter.
""")

def proses_command(user_input, messages):
    """Memproses command. Mengembalikan "keluar", "selesai" (command sudah ditangani), atau None (bukan command)."""
    perintah = user_input.lower()

    if perintah == "exit":
        print("👋 Terima kasih telah menggunakan InternLens. Sampai jumpa!")
        return "keluar"

    if perintah == "clear":
        reset_riwayat(messages)
        print("🧹 Riwayat dihapus. Percakapan baru dimulai.")
        return "selesai"

    if perintah == "save":
        simpan_riwayat(messages)
        return "selesai"

    if perintah == "help":
        tampilkan_bantuan()
        return "selesai"

    if perintah.startswith("load"):
        bagian = user_input.split(maxsplit=1)
        if len(bagian) < 2:
            print("Format: load nama_file.json")
        else:
            data = muat_riwayat(bagian[1])
            if data:
                messages.clear()
                messages.extend(data)
                print(f"📂 Riwayat dimuat ({len(messages)} pesan).")
        return "selesai"

    return None   # bukan command, berarti pesan biasa untuk chatbot

## Step 10: Main Chatbot Loop

**Tujuan:** Menyatukan semua bagian menjadi chatbot yang berjalan berulang.

**Cara kerja:** Loop membaca input, memeriksa apakah itu command, lalu mengirimnya ke `proses_pesan`. Fungsi ini menambahkan pesan user, memanggil streaming, dan menambahkan jawaban ke riwayat hanya jika request berhasil. Chatbot interaktif dijalankan di cell paling akhir; untuk pengujian Step 11–13 cukup memanggil `proses_pesan` langsung.

**Hubungannya dengan tugas:** Conversation history (pola `append` user lalu assistant), memori dalam satu sesi, dan error handling (jawaban gagal tidak disimpan).

In [ ]:
# 10. Main chatbot loop
def proses_pesan(messages, user_input):
    """Satu putaran percakapan: tambah pesan user, kirim ke LLM, simpan jawaban ke riwayat."""
    messages.append({"role": "user", "content": user_input})

    print("\nInternLens:")
    jawaban = kirim_pesan_streaming(messages)

    if jawaban is None:
        # Request gagal: buang pesan user agar riwayat tetap bersih
        messages.pop()
    else:
        messages.append({"role": "assistant", "content": jawaban})

def main():
    print(BANNER)
    tampilkan_bantuan()

    while True:
        user_input = input("\nKamu: ").strip()
        if not user_input:
            continue

        if user_input.lower() == "contoh":
            user_input = CONTOH_JOB_DESC
            print("(Mengirim contoh job description Data Analyst Intern...)")
        else:
            status = proses_command(user_input, messages)
            if status == "keluar":
                break
            if status == "selesai":
                continue

        proses_pesan(messages, user_input)

## Step 11: Testing dengan Contoh Internship Data Analyst

**Tujuan:** Memastikan format analisis dan sikap netral chatbot sesuai rancangan.

**Cara kerja:** Contoh job description dikirim tanpa profil user.

**Yang perlu diperiksa pada output:**
- Struktur analisis (Role Overview sampai Overall Takeaway) muncul.
- Scope yang luas, requirement yang tidak selaras, allowance, supervisor, dan detail jadwal/work arrangement disebut sebagai *concern* atau *point to clarify*, bukan tuduhan.
- Bagian Skill Match meminta user memberikan profil.
- Tidak ada skor atau kata "scam".

In [ ]:
# 11. Test: contoh job description Data Analyst Intern
reset_riwayat(messages)
proses_pesan(messages, CONTOH_JOB_DESC)

## Step 12: Testing Conversation Memory

**Tujuan:** Membuktikan chatbot mengingat konteks sebelumnya.

**Yang perlu diperiksa:** Bagian Skill Match pada jawaban kedua memakai Python, SQL, Pandas, Power BI, dan seterusnya dari pesan pertama, dengan kategori Strong Match, Partial Match, Skill Gap, dan Need Clarification (misalnya Photoshop dan video editing masuk Skill Gap). Jumlah pesan harus 5, dan file JSON muncul di panel file Colab.

In [ ]:
# 12. Test: conversation memory
reset_riwayat(messages)

# Pesan 1: profil user
proses_pesan(messages, "Saya mahasiswa Sains Data. Skill saya Python, SQL, Pandas, Scikit-learn, Power BI, dan basic statistics.")

# Pesan 2: job description (tanpa mengulang profil)
proses_pesan(messages, CONTOH_JOB_DESC)

# Verifikasi isi riwayat: system + 2 user + 2 assistant = 5 pesan
print(f"\nJumlah pesan dalam riwayat: {len(messages)}")

# Sekaligus uji fitur save
simpan_riwayat(messages)

## Step 13: Testing Error Handling

**Tujuan:** Memastikan program tidak crash dan riwayat tidak rusak saat API gagal.

**Yang perlu diperiksa:** Muncul pesan `⚠️ Maaf, terjadi masalah...`, program tetap berjalan, dan jumlah pesan sebelum dan sesudah sama (pesan user yang gagal dibuang). `MODEL_NAME` dikembalikan otomatis di akhir cell.

In [ ]:
# 13. Test: error handling
reset_riwayat(messages)
jumlah_awal = len(messages)

# Simulasi error: sementara memakai nama model yang tidak ada
MODEL_ASLI = MODEL_NAME
MODEL_NAME = "model-tidak-ada"

proses_pesan(messages, "Halo InternLens")

# Kembalikan model semula
MODEL_NAME = MODEL_ASLI

print(f"\nJumlah pesan sebelum: {jumlah_awal}, sesudah error: {len(messages)}")

## Step 14: Checklist Requirement Tugas

| Requirement | Lokasi di kode |
|---|---|
| Berbasis LLM, Groq API, bukan model lokal | Step 3 (`Groq`, `MODEL_NAME`) |
| Berjalan di Google Colab/console | Seluruh notebook, `main()` di Step 10 |
| System prompt | Step 4 (`SYSTEM_PROMPT`) |
| Conversation history dan konteks satu sesi | Step 5, Step 10 (`proses_pesan`), dibuktikan di Step 12 |
| Error handling | Step 6, Step 7 (`try/except`), dibuktikan di Step 13 |
| Minimal 2 command | Step 9 (`exit`, `clear`, `save`, `help`) |
| API key tidak di-hardcode | Step 2 (Colab Secrets, fallback `getpass`) |
| Bonus: streaming | Step 7 (`kirim_pesan_streaming`) |
| Bonus: save history | Step 8 (`simpan_riwayat`) dan command `save` |
| Fitur tambahan relevan tema | Red Flag Scan, Skill Match, Questions to Ask Recruiter (system prompt), plus `load` dan `contoh` |

## 🚀 Jalankan InternLens (Interaktif)

Jalankan cell di bawah, lalu ketik pesan di kolom input. Ketik `help` untuk melihat daftar command dan `exit` untuk keluar.

Contoh alur: (1) ceritakan skill kamu, (2) tempel job description, atau ketik `contoh` untuk demo.

In [ ]:
# Chatbot interaktif
reset_riwayat(messages)
main()